# C1.3 · Attacking evaluation itself

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Security of AI*

Builds on **[C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**.

| | |
|---|---|
| Tools used | Cyber Commons eval harness, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Game the B2.14 scoring harness deliberately, then close the hole you used.

**Why a security engineer needs it.** If the eval can be fooled, the assurance is theatre. The control it builds is: eval gaming, sandbagging, contamination and judge manipulation as test cases.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Your evaluation is a control, and controls get attacked. A benchmark with a leaked key, a skewed class balance or a scorer that can be satisfied without solving anything is a control that reports itself green forever.

> **At CyberTravels.** The benchmark that says CyberTravels' review harness scores 0.9 is itself a control, and it gets attacked. A leaked key or a loose matcher makes it report green forever.

## 2 · The framework

```
   the benchmark is a control, so attack it

   leaked key      -> the score is a training metric
   skewed classes  -> a constant answer scores 0.875
   loose matching  -> wrong answers match on basename
   gameable oracle -> satisfied without solving anything

   an unattacked evaluation reports itself green forever
```

If you can make a harness score well without being good, so can the vendor whose
benchmark you are reading — and so can your own team, without meaning to.

Three exploits work on almost every published security-harness result:

1. **Report conformance as quality.** Schema validity is ~100% by construction
   with structured output. It measures nothing about correctness (B2.14).
2. **Exploit class imbalance.** If 80% of a corpus is one CWE, always guessing
   that CWE scores 0.8 with no capability at all.
3. **Exploit basename collisions.** If the matcher compares bare filenames and
   the corpus reuses `1.py` across directories, scores become partly random —
   and random noise on a leaderboard looks like a small improvement.

Red-teaming evaluation means running these three against your own numbers before
someone else does.

## 3 · The procedure, as a skill

Before believing an evaluation, score a harness with no capability at all. The skill does exactly that against the skewed corpus, then rebalances and re-scores — and separately checks whether the matcher rewards answers naming the wrong directory.

### The skill — [`skills/redteam/eval-corpus-integrity-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/eval-corpus-integrity-check/SKILL.md)

```yaml
name: eval-corpus-integrity-check
description: >-
  Attack an evaluation rather than a system: score a deliberately
  zero-capability harness against the corpus and see what the corpus alone
  awards it, then rebalance and re-score. Use before trusting any benchmark
  number, your own included.
allowed-tools: Read, Grep, Glob
```

# Score the null harness first

An evaluation is a claim about a system, and it is only as good as the corpus
underneath it. The cheapest way to find out is to score a harness that has no
capability at all — one that answers from the corpus's own skew. A high score
there is a measurement of the corpus, and every number derived from it inherits
the problem.

## When to use this

Before publishing an evaluation, before believing one, and whenever a benchmark
result is surprisingly good.

## Procedure

**1 — Build the null harness.** No analysis: answer with whatever the corpus's
majority class is. It should be trivial to write, and writing it takes minutes.

**2 — Score it against the corpus as it stands.** Record conformance and
accuracy separately — a null harness usually conforms perfectly, which is itself
worth knowing about conformance as a metric.

**3 — Rebalance and re-score.** Equalise the classes and run the same null
harness. The drop is the portion of the original score that came from skew
rather than capability.

**4 — Check the matcher, not just the corpus.** Score answers that name the
wrong directory. If they still score, the matcher is measuring string similarity
rather than correctness, and it will flatter every harness equally.

**5 — Report the null score alongside every real one.** A harness scoring 0.85
against a null of 0.85 has demonstrated nothing, and that comparison is the only
honest way to read the number.

## Output contract

```json
{
  "corpus": {"n": 0, "skew": 0.0, "classes": {"str": 0}},
  "null_harness": {"conformance": 0.0, "accuracy_skewed": 0.0, "accuracy_balanced": 0.0},
  "matcher": {"kind": "str", "wrong_directory_scores": 0.0},
  "verdict": {"score_from_skew": 0.0, "score_from_capability": 0.0}
}
```

## Failure modes

- **Reporting accuracy without the null baseline.** It is unreadable without
  one.
- **Conflating conformance with accuracy.** The null harness conforms.
- **Trusting a matcher that has never been attacked.** Try to fool it before you
  publish with it.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/eval-corpus-integrity-check/scripts/eval_corpus_integrity_check.py
SCRIPT = "skills/redteam/eval-corpus-integrity-check/scripts/eval_corpus_integrity_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

On the skewed corpus the zero-capability harness scores conformance 1.00 and expert accuracy around 0.85. Balancing the corpus drops expert accuracy to roughly 0.25. Answers naming the wrong directory score near zero under `path_key` and near 1.00 under a basename matcher. The audit reports the majority-class share, the trivial baseline, the matcher collisions, and near-zero lift.

## Your turn

Run the audit against a benchmark result your organisation relies on. Two questions decide it: what does always guessing the majority class score, and does the matcher collide? Most published numbers answer neither.

---

**Next → [C1.4 · Reporting agentic findings](https://spbreed.github.io/cyber-commons/lessons/C1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*